

> Añadir blockquote
importamos librerias


In [ ]:
!pip install datasets==3.6.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 8.8 MB/s eta 0:00:00
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [8]:
!git clone https://github.com/CCogS-Mx/text-preprocessing.git
!git clone https://github.com/CCogS-Mx/fine-tuning-bert-and-friends.git
import sys
sys.path.append('./text-preprocessing/scripts/')

fatal: destination path 'text-preprocessing' already exists and is not an empty directory.
fatal: destination path 'fine-tuning-bert-and-friends' already exists and is not an empty directory.


In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer
from datasets import load_dataset
corpus=load_dataset("imdb")
from sklearn.model_selection import train_test_split

from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV


from sklearn.metrics import classification_report

import pandas as pd
import numpy as np
import text_preprocessing as tp



[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


2. ANALISAMOS LOS DATOS

In [14]:
data = pd.DataFrame(corpus['train'])
data.head()

,text,label
0,I rented I AM CURIOUS-YELLOW from my video sto...,0
1,"""I Am Curious: Yellow"" is a risible and preten...",0
2,If only to avoid making this type of film in t...,0
3,This film was probably inspired by Godard's Ma...,0
4,"Oh, brother...after hearing about this ridicul...",0


In [17]:
train_data,test_data = train_test_split(data, test_size=0.2,random_state=42)

In [18]:
test_data.head()

,text,label
6868,"Dumb is as dumb does, in this thoroughly unint...",0
24016,I dug out from my garage some old musicals and...,1
9668,After watching this movie I was honestly disap...,0
13640,This movie was nominated for best picture but ...,1
14018,Just like Al Gore shook us up with his painful...,1


In [19]:
test_data.tail()

,text,label
8670,If you seen Rodney Dangerfield's previous movi...,0
11839,Anything that might have been potentially inte...,0
4013,That is quite an outdated movie which aims to ...,0
21147,"This is a comedy of morals, so occasionally a ...",1
695,When I started watching this movie I saw the d...,0


In [20]:
preprocessing = tp.Preprocessing('english')


In [21]:
from pickle import TRUE
from os import remove
train_data_preprocessed = preprocessing.main_preprocess(train_data,'text',remove_stop_words=True)
test_data_preprocessed = preprocessing.main_preprocess(test_data,'text',remove_stop_words=True)

In [22]:
train_data_preprocessed.head()

,text,label
23311,borrowed movie despite extremely low rating wa...,1
23623,unexpected accident killed inexperienced climb...,1
1020,summer blockbuster hit baseketball this one mo...,0
12645,scarcely imagine better movie thishey go chick...,1
1533,still famous decadent actor morgan freeman fil...,0


In [23]:
Tfidf_Vectorizer= TfidfVectorizer(ngram_range=(1,2))

In [26]:
#Tfidf_Vectorizer.fit(train_data_preprocessed['text'])#modelo de bolsa de palabra
#Tfidf_Vectorizer.transform(train_data_preprocessed['text'])
Tfidf_Vectorizer.fit_transform(test_data_preprocessed['text'])


<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 1133783 stored elements and shape (5000, 502698)>

In [27]:
x_test = Tfidf_Vectorizer.transform(test_data_preprocessed['text'])
#x_train = Tfidf_Vectorizer.transform(train_data_preprocessed['text'])

In [38]:
x_train_tfidf=Tfidf_Vectorizer.transform(test_data_preprocessed['text'])

In [37]:
print(x_train_tfidf)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 1133783 stored elements and shape (5000, 502698)>
  Coords	Values
  (0, 17071)	0.08668510521025802
  (0, 17072)	0.09086003101542524
  (0, 45372)	0.05583914652200405
  (0, 45376)	0.07954802240577005
  (0, 46581)	0.11811251421680843
  (0, 46622)	0.2147603074155725
  (0, 48690)	0.057116471238175384
  (0, 48723)	0.07954802240577005
  (0, 57736)	0.14317353827704835
  (0, 57746)	0.09086003101542524
  (0, 57747)	0.09086003101542524
  (0, 61486)	0.04397042752523772
  (0, 61565)	0.09086003101542524
  (0, 68522)	0.02715231736330332
  (0, 69224)	0.08668510521025802
  (0, 73867)	0.05751254445198676
  (0, 73881)	0.09086003101542524
  (0, 81073)	0.10141100570466224
  (0, 81188)	0.08668510521025802
  (0, 81533)	0.18172006203085048
  (0, 101839)	0.11906304773448925
  (0, 101943)	0.0837229482109373
  (0, 101979)	0.09086003101542524
  (0, 102011)	0.0837229482109373
  (0, 116856)	0.05078738193480138
  :	:
  (4999, 416886)	0.08744840438101859
 

In [39]:
smv = LinearSVC(random_state=42)
smv.fit(x_train_tfidf,test_data_preprocessed['label'])


LinearSVC(random_state=42)

In [44]:
preds=smv.predict(x_train_tfidf)
print(classification_report(test_data_preprocessed['label'],preds))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      2515
           1       1.00      1.00      1.00      2485

    accuracy                           1.00      5000
   macro avg       1.00      1.00      1.00      5000
weighted avg       1.00      1.00      1.00      5000

